In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [5]:
# =========================
# 1. CSV 로드
# =========================
csv_path = "/Users/jang-wonjun/Desktop/DMISLab/ToxAgent/molecule_safe_ver/splits/scaffold_by_endpoint/split_summary.csv"   # <-- 파일명으로 바꿔줘
df_stats = pd.read_csv(csv_path)

# 숫자형 컬럼 보정
num_cols = ["n_total", "n_valid_smiles", "n_train", "n_valid", "n_test"]
for col in num_cols:
    df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce").fillna(0).astype(int)

print("데이터 미리보기")
print(df_stats.head(), "\n")

데이터 미리보기
  dataset_name  endpoint    smiles_col  n_total  n_valid_smiles  n_train  \
0         ames      ames  toxic_smiles      109             109       87   
1      clintox   clintox  toxic_smiles        5               5        4   
2     dictrank  dictrank  toxic_smiles        5               5        4   
3       dilist    dilist  toxic_smiles       10              10        8   
4        diril     diril  toxic_smiles        1               1        0   

   n_valid  n_test  note  
0       11      11   NaN  
1        0       1   NaN  
2        0       1   NaN  
3        1       1   NaN  
4        0       1   NaN   



/var/folders/r2/hf806xw17vv9p1hn3v10qr980000gn/T/ipykernel_17589/116352180.py:10: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_stats[col] = pd.to_numeric(df_stats[col], errors="coerce").fillna(0).astype(int)
/var/folders/r2/hf806xw17vv9p

In [13]:
total_test_num = 0
for n_test_num in df_stats['n_test']:
    total_test_num += n_test_num
total_test_num

693

In [6]:
# =========================
# 2. dataset별 요약 통계
# =========================
dataset_summary = (
    df_stats.groupby("dataset_name", as_index=False)
    .agg(
        n_endpoints=("endpoint", "count"),
        total_samples=("n_total", "sum"),
        total_train=("n_train", "sum"),
        total_valid=("n_valid", "sum"),
        total_test=("n_test", "sum"),
    )
    .sort_values("total_samples", ascending=False)
)

print("=== Dataset별 요약 ===")
print(dataset_summary.to_string(index=False), "\n")

=== Dataset별 요약 ===
 dataset_name  n_endpoints  total_samples  total_train  total_valid  total_test
 herg_central            1           4641         3712          464         465
   herg_karim            1           1369         1095          137         137
     tox21_df           12            318          250           31          37
  sider_train           27            234          177           23          34
         ames            1            109           87           11          11
         herg            1             26           20            3           3
       dilist            1             10            8            1           1
      clintox            1              5            4            0           1
     dictrank            1              5            4            0           1
skin_reaction            1              2            0            0           2
        diril            1              1            0            0           1 



In [12]:
# =========================
# 3. endpoint별 요약 통계
# =========================
endpoint_summary = (
    df_stats[["dataset_name", "endpoint", "n_total", "n_train", "n_valid", "n_test"]]
    .sort_values(["n_total", "dataset_name"], ascending=[False, True])
)

print("=== Endpoint별 요약 (n_total 큰 순) ===")
print(endpoint_summary.to_string(index=False),"\n")

=== Endpoint별 요약 (n_total 큰 순) ===
 dataset_name                                                            endpoint  n_total  n_train  n_valid  n_test
 herg_central                                                          hERG_inhib     4641     3712      464     465
   herg_karim                                                          herg_karim     1369     1095      137     137
         ames                                                                ames      109       87       11      11
     tox21_df                                                         tox21_NR-ER       56       44        6       6
     tox21_df                                                        tox21_SR-MMP       50       40        5       5
     tox21_df                                                     tox21_NR-ER-LBD       39       31        4       4
     tox21_df                                                        tox21_NR-AhR       33       26        3       4
     tox21_df                

In [8]:
# =========================
# 4. 작은 endpoint 개수 확인
# =========================
bins_summary = {
    "n_total < 5": (df_stats["n_total"] < 5).sum(),
    "5 <= n_total < 10": ((df_stats["n_total"] >= 5) & (df_stats["n_total"] < 10)).sum(),
    "10 <= n_total < 30": ((df_stats["n_total"] >= 10) & (df_stats["n_total"] < 30)).sum(),
    "30 <= n_total < 100": ((df_stats["n_total"] >= 30) & (df_stats["n_total"] < 100)).sum(),
    "n_total >= 100": (df_stats["n_total"] >= 100).sum(),
}
print("=== Endpoint 크기 구간별 개수 ===")
for k, v in bins_summary.items():
    print(f"{k}: {v}")
print()

=== Endpoint 크기 구간별 개수 ===
n_total < 5: 5
5 <= n_total < 10: 16
10 <= n_total < 30: 19
30 <= n_total < 100: 5
n_total >= 100: 3



In [10]:
# =========================
# 6. imbalance 확인용 비율 컬럼
# =========================
df_stats["train_ratio"] = df_stats["n_train"] / df_stats["n_total"].replace(0, 1)
df_stats["valid_ratio"] = df_stats["n_valid"] / df_stats["n_total"].replace(0, 1)
df_stats["test_ratio"]  = df_stats["n_test"]  / df_stats["n_total"].replace(0, 1)

print("=== Split ratio 예시 ===")
print(
    df_stats[
        ["dataset_name", "endpoint", "n_total", "train_ratio", "valid_ratio", "test_ratio"]
    ].sort_values("n_total", ascending=False).head(20).to_string(index=False)
)

=== Split ratio 예시 ===
dataset_name                                        endpoint  n_total  train_ratio  valid_ratio  test_ratio
herg_central                                      hERG_inhib     4641     0.799828     0.099978    0.100194
  herg_karim                                      herg_karim     1369     0.799854     0.100073    0.100073
        ames                                            ames      109     0.798165     0.100917    0.100917
    tox21_df                                     tox21_NR-ER       56     0.785714     0.107143    0.107143
    tox21_df                                    tox21_SR-MMP       50     0.800000     0.100000    0.100000
    tox21_df                                 tox21_NR-ER-LBD       39     0.794872     0.102564    0.102564
    tox21_df                                    tox21_NR-AhR       33     0.787879     0.090909    0.121212
    tox21_df                                    tox21_SR-ARE       32     0.781250     0.093750    0.125000
    t

/var/folders/r2/hf806xw17vv9p1hn3v10qr980000gn/T/ipykernel_17589/1165544243.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_stats["train_ratio"] = df_stats["n_train"] / df_stats["n_total"].replace(0, 1)
/var/folders/r2/hf806xw17vv9p1h